# D04这份数据是谁

In [ ]:
%reload_ext autoreload
%autoreload 2 
# 上面这俩用于自动重载

import sqlite3 # 先导入内置的模块

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from signal_insight import config, quality, viz, geo
from signal_insight.viz import (MAIN,MAIN_LIGHT,MAIN_DARK,MAIN_MID,GRAY,
                    GRAY_DARK,RED,AMBER,INK,MUTED,FAINT) # 配色常量

con = sqlite3.connect(config.DB_PATH) # 连接数据库
DAY = config.ANALYSIS_DAY
NB = "D03_信令数据洞察"
SOURCE = f"{config.SOURCE_NOTE}:脚本:{NB} 制图:{config.AUTHOR}"

## 封装测试

### 点落面封装测试

In [ ]:
cell = pd.read_sql("SELECT * FROM ods_cell", con)
sites = cell[['lng', 'lat']].drop_duplicates().reset_index(drop=True)
# 给站点做编号：编号补0到S00005
sites['site_id'] = "S" + (sites.index + 1).astype(str).str.zfill(5)

districts = geo.rings_from_points(
    pd.read_sql("SELECT * FROM ods_district_boundary", con), "district")
sites['districts'] = geo.point_in_polygons(sites['lng'], sites['lat'], districts)
sites['districts'] = sites['districts'].fillna("市外")


# 没坐标点扇区定位不了
cell = cell.dropna(subset=['lng','lat']).copy()
# 不重复的坐标=一个站点
# 把站点号贴回每个扇区
cell = cell.merge(sites, on=['lng', 'lat'],how='left')

### 区域地图封装测试

In [ ]:
# 读取dwd表
dwd = pd.read_sql("SELECT * FROM dwd_signal", con, parse_dates=['ts'])
# 两个表格合并
joined = dwd.merge(cell[['lac_cell', "site_id", 'lng', 'lat']],
                on='lac_cell',
                how='left')
out_share = (sites['districts'] == '市外').mean()
rec_district = joined.merge(sites[['site_id', 'districts']],
            on="site_id",how="left")
rec_out = (rec_district['districts'] =='市外').mean()

v2_pts = pd.read_sql("SELECT * FROM ods_aoi_v2", con)
aoi_v2 = geo.rings_from_points(v2_pts,"name")

sites['aoi'] = geo.point_in_polygons(sites['lng'],sites['lat'], aoi_v2)
dim_cell = cell.merge(sites[['site_id', 'aoi']],
                      on='site_id',
                      how='left')
def split_at_jump(values: pd.Series) -> tuple[list[str], list[str], float]:
    """按数值从小到大排，找相邻两项比值最大的地方切开。
    返回 (断崖之前的名字, 断崖之后的名字, 断崖处的倍数)。D3_02、D3_03 用它决定强调哪一组。"""
    v = values.sort_values()
    r = v.values[1:] / v.values[:-1]
    k = int(r.argmax())
    return list(v.index[: k + 1]), list(v.index[k + 1:]), float(r[k])

site = dim_cell.groupby('site_id').agg(
    lng=("lng", 'first'),
    lat=('lat', 'first'),
    district=('districts', 'first')
)

In [ ]:
from scipy.spatial import cKDTree # pip install scipy
# ============================================================
# 2. 经纬度 → 平面公里（沈阳局部平面近似）
# ============================================================
lng0, lat0 = 123.4, 41.8
kx = 111.320 * np.cos(np.radians(lat0))   # 每度经度 ≈ 多少公里
ky = 110.574                               # 每度纬度 ≈ 多少公里
xy = np.column_stack([
    (site["lng"] - lng0) * kx,
    (site["lat"] - lat0) * ky,
]).astype(np.float64)   # cKDTree 用 float64 更稳
# ============================================================
# 3. KD 树求最近邻（等价于原来的分块暴力 + 排除自己）
# ============================================================
tree = cKDTree(xy)
# k=2：第 0 列是自己（距离 0），第 1 列是真正的最近邻
dist_km, idx = tree.query(xy, k=2)
# 取第 1 列作为最近邻距离（公里），转成米
nn = dist_km[:, 1].astype(np.float32)
site["nn_m"] = nn * 1000
# ============================================================
# 4. 各区县面积（鞋带公式，与原代码一致）
# ============================================================
area = {}
for name, rings in districts.items():
    area[name] = sum(
        0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
        for x, y in (((r[:, 0] - lng0) * kx, (r[:, 1] - lat0) * ky) for r in rings)
    )
# ============================================================
# 5. 分组统计
# ============================================================
by = (site[site["district"] != "市外"]
        .groupby("district")
        .agg(站点数=("nn_m", "size"), 间距中位_米=("nn_m", "median"))
        .sort_values("间距中位_米"))
by["每平方公里站点数"] = by["站点数"] / pd.Series(area)
by = by.round(1).reset_index()

## 用户都去过那些区

In [ ]:
%reload_ext autoreload
%autoreload 2 
# 上面这俩用于自动重载

import sqlite3 # 先导入内置的模块

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from signal_insight import config, quality, viz, geo
from signal_insight.viz import (MAIN,MAIN_LIGHT,MAIN_DARK,MAIN_MID,GRAY,
                    GRAY_DARK,RED,AMBER,INK,MUTED,FAINT) # 配色常量
fonts = viz.apply_theme("default")         
con = sqlite3.connect(config.DB_PATH) # 连接数据库
DAY = config.ANALYSIS_DAY
NB = "D03_信令数据洞察"
SOURCE = f"{config.SOURCE_NOTE}:脚本:{NB} 制图:{config.AUTHOR}"

districts = geo.rings_from_points(
    pd.read_sql("SELECT * FROM ods_district_boundary", con),"district"
)

aoi = geo.rings_from_points(
   pd.read_sql("SELECT * FROM ods_aoi_v2", con),"name" 
)
print(f"一共有{len(districts)}个区县,{len(aoi)}个商圈")

In [ ]:
dwd =pd.read_sql("SELECT imsi, ts, lac_cell FROM dwd_signal", con)
dim = pd.read_sql("SELECT * FROM dim_cell", con)

sig = dwd.merge(dim, on=['lac_cell'], how='inner')
n_users = sig['imsi'].nunique()
print(f"带位置的信令有：{len(sig)}条,用户有：{n_users}个")
# 需求：10月3日这一天，有多少比例的用户到过每个区
reach = (sig.groupby('district')['imsi'].nunique() / n_users).rename('到访用户占比')
reach = reach.drop("市外", errors='ignore').sort_values().reset_index()
reach.to_sql('ads_district_rech',con,if_exists='replace', index=False)

In [ ]:
reach['到访用户占比'] = reach['到访用户占比']  * 100

In [ ]:
s = reach.sort_values(by='到访用户占比',ascending=False)   # 大 → 小，条形从上往下就是这个顺序
s.set_index('district',inplace=True)
FAR = ["新民市", "辽中区", "法库县", "康平县"]      # 远郊四区县 = 排序后垫底的四个

top, bottom = s.index[0], s.index[-1]
ratio = s.iloc[0] / s.iloc[-1]                      # 最高 ÷ 最低
cut = s.index.get_loc(FAR[0])                       # 远郊段从第几行开始
jump = s.iloc[cut - 1] / s.iloc[cut]                # 断档处掉了几倍

def pct(v: float) -> str:
    """百分比写法（规范第 6 节）：一般取整，小于 10 留一位，小于 1 留两位——很小的值不写成 0%。"""
    return f"{v:.0f}%" if v >= 10 else f"{v:.1f}%" if v >= 1 else f"{v:.2f}%"

# ── 2 版面：左地图 + 右条形 ────────────────────────────────────────────────
regions = viz.maps.Regions(districts)
H, BAR_W = 10, 5.6                                 # 地图高、条形列宽（英寸）
MAP_W = H * regions.aspect + 3.4                    # 地图本身 + 两侧引线标注
fig, (ax_map, ax_bar) = plt.subplots(
    1, 2, figsize=(12, 6),
    width_ratios=[MAP_W, BAR_W], layout="constrained")

# ── 3 左图：填色 + 引线标注 ───────────────────────────────────────────────
# 占比跨两个数量级（0.3% → 97%），填色用对数色阶；没数据的区露出底色
regions.base(ax_map, scale=False, north=dict(x=0.93, y=0.95))
regions.fill(ax_map, dict(s.reset_index().values.tolist()), norm="log")
regions.edges(ax_map, FAR, color=MAIN, lw=0.9)      # 远郊四区县整圈描蓝，和右图对上
regions.label(ax_map, dict(s.reset_index().values.tolist()), fmt=pct, hot=FAR,  split='balance')

# ── 4 右图：横向条形，远郊四区县用主色 ────────────────────────────────────
y = np.arange(len(s))
ax_bar.barh(y, s.iloc[:,0].values, height=0.62,
            color=[MAIN if n in FAR else GRAY for n in s.index])
for i, (name, v) in enumerate(zip(s.index, s.iloc[:,0].values)):
    ax_bar.text(v + 1.2, i, pct(v), va="center", fontsize=10,
                color=MAIN if name in FAR else INK,
                fontweight="bold" if name in FAR else "normal")

ax_bar.set_yticks(y, s.index, fontsize=10.5)
ax_bar.set_ylim(len(s) - 0.4, -1.4)                 # 顶上留一行放列头
ax_bar.set_xlim(0, 118)                             # 条形从 0 开始，右边留出数值的位置
ax_bar.set_xticks([])
ax_bar.grid(False)
ax_bar.tick_params(length=0)
for sp in ax_bar.spines.values():
    sp.set_visible(False)
ax_bar.text(0, -1.3, "到访用户占比（分母 = 全部用户）", 
        fontsize=10.5, color=INK, va="center")

# 断档线：掉到 1/jump 的那一行上面
ax_bar.axhline(cut - 0.5, color=MUTED, lw=0.8, 
            ls=(0, (3, 3)), xmin=0.02, xmax=0.98)
ax_bar.text(118, cut - 0.62, f"此处降到 1/{jump.values[0]:.1f}", ha="right", va="bottom",
            fontsize=9.5, color=MAIN)

# ── 5 标题区、来源行、保存 ────────────────────────────────────────────────
viz.finish(
    fig,
    f"{top}被 {pct(s.iloc[0,0])} 的用户到访，{bottom}只有 {pct(s.iloc[-1,0])}，相差 {ratio.values[0]:.0f} 倍",
    "左图填色越深到访用户越多（对数色阶），区名后为到访用户占比；右图按占比从大到小排序"
    f"蓝色 = 远郊四区县\n"
    "一个用户可出现在多个区，各区分别计数，合计远超 100%，不能相加",
    f"数据来源：沈阳移动信令样本（脱敏，2018 年国庆）；制图:{config.AUTHOR}",
    "D3_04_区县到访占比.png",
)
plt.show()

::: SUMMARY 结论
10月3日，97%的样本都到过和平区，其他区县的到访比例在57%以下。分布明显集中于和平区，说明样本样本并非全市随机抽样，结论不能推广到全市。所以需要限定数据分析范围。
:::

::: TIP 提示：大模型作图注意事项
- 数据不要无脑发：发前几行，大致描述下数据，什么类型，series和datafarme数据差的很远
- 不要发全量数据
- 发的时候：把你的本地包可以打成压缩包发给他，或者给它发包的文档.
:::

## 热点在哪里?

需求：用户都集中去了和平区的哪里？找出连接最多的基站，看看附近是什么地方

In [ ]:
site_users = sig.groupby(
    ['site_id', 'lng', 'lat'])['imsi'].nunique() \
    .rename("用户数").reset_index()  \
    .sort_values("用户数",ascending=False,ignore_index=True)
site_users['用户占比'] = site_users['用户数'] / n_users
site_users.head(5)

top5 = site_users.head(5)
hot_lng = np.average(top5['lng'], weights=top5['用户数'])
hot_lat = np.average(top5['lat'], weights=top5['用户数'])
poi = pd.read_sql("SELECT * FROM ods_poi_purpose",con)
poi['距热点(米)'] = geo.haversine(poi['lat'],poi['lng'],hot_lat,hot_lng).round()
poi.nsmallest(6,"距热点(米)")[['name','type','距热点(米)']]

dim_hotspot = pd.DataFrame([{
    "name": "西塔", "lng": hot_lng, "lat":hot_lat, "radius_m": 500,
    "依据": "连接用户数前5名的站点按用户数加权的重心，附近为韩百商场等一带"
}])
dim_hotspot.to_sql("dim_hotspot", con, if_exists="replace", index=False)
dim_hotspot

```text
site_users构建过程：
site_users = sig.groupby(
    ['site_id', 'lng', 'lat'])['imsi'].nunique() \
    .rename("用户数").reset_index()  \
    .sort_values("用户数",ascending=False,ignore_index=True)
site_users['用户占比'] = site_users['用户数'] / n_users
输出前5行数据如下
print(site_users.head().to_markdown()) # site_users是个dataframe
|    | site_id   |     lng |     lat |   用户数 |   用户占比 |
|---:|:----------|--------:|--------:|---------:|-----------:|
|  0 | S07814    | 123.409 | 41.8089 |     2300 |   0.407512 |
|  1 | S07823    | 123.412 | 41.8079 |     1769 |   0.31343  |
|  2 | S07364    | 123.413 | 41.8047 |     1648 |   0.291991 |
|  3 | S07363    | 123.411 | 41.8059 |     1633 |   0.289334 |
|  4 | S13669    | 123.409 | 41.8064 |     1146 |   0.203047 |

继续处理拿到如下信息：
top5 = site_users.head(5)
hot_lng = np.average(top5['lng'], weights=top5['用户数'])
hot_lat = np.average(top5['lat'], weights=top5['用户数'])
poi = pd.read_sql("SELECT * FROM ods_poi_purpose",con)
poi['距热点(米)'] = geo.haversine(poi['lat'],poi['lng'],hot_lat,hot_lng).round()
poi.nsmallest(6,"距热点(米)")[['name','type','距热点(米)']]

print(poi.nsmallest(6,"距热点(米)")[
    ['name','type','距热点(米)']].to_markdown())

|      | name               | type   |   距热点(米) |
|-----:|:-------------------|:-------|-------------:|
|  973 | 韩百商场           | 购物   |           17 |
| 1116 | 韩百商场A座        | 购物   |           47 |
| 1049 | 博曼广场           | 购物   |          168 |
|  687 | 安图社区           | 居住   |          230 |
| 1050 | 鑫韩城购物广场     | 购物   |          257 |
| 1007 | 沈阳朝鲜族百货大楼 | 购物   |          273 |

左边地图，局部 放大放到右边，然后显示该区域的基站分布，其中连接的比较大的用大的点，类似气泡图再平面地图上，

dim_hotspot = pd.DataFrame([{
    "name": "西塔", "lng": hot_lng, "lat":hot_lat, "radius_m": 500,
    "依据": "连接用户数前5名的站点按用户数加权的重心，附近为韩百商场等一带"
}])

绘制一个圈标记西塔这个地区周边500米，实际上几个热点区域就是再西塔一带附近
```

In [ ]:
# ── D4_02：内五区热力 + 西塔放大；hb 保留给下一格检查 ──
core_names = ['铁西区', '和平区', '沈河区', '皇姑区', '大东区']
core = viz.maps.Regions({n: districts[n] for n in core_names})
hs = dim_hotspot.loc[dim_hotspot['name'].eq('西塔')].iloc[0]
lng, lat, radius_m = (float(hs['lng']), float(hs['lat']), float(hs['radius_m']))
sites = site_users.sort_values('用户数', ascending=False).copy()
top = sites.head(5)
distances = geo.haversine(sig['lat'], sig['lng'], lat, lng)
reach_n = sig.loc[distances <= radius_m, 'imsi'].nunique()
reach_pct = reach_n / sig['imsi'].nunique()
top_in = int((geo.haversine(top['lat'], top['lng'], lat, lng) <= radius_m).sum())
fig, (amap, az) = plt.subplots(1, 2, figsize=(13, 7.8),
                               width_ratios=[core.aspect * 1.06, 1], layout='constrained')
core.base(amap, north=dict(x=0.94, y=0.94))
hb = core.hexbin(amap, sig, 900, value='imsi', how='nunique', min_value=5)
core.text(amap, dict.fromkeys(core_names, np.nan), alias={'沈河区': '沈河'}, overflow='hide')
viz.maps.colorbar(hb, amap, label='格内到过的用户（去重，人；格宽 900 米）')
amap.set_title('A  内五区：西塔在城市中的位置', loc='left', fontsize=11, pad=12)
core.base(az, fill=False, outline=False, north=False,
          borders=dict(color='#C4CDD8', lw=0.8, ls=(0, (4, 3))))
viz.maps.zoom(amap, az, lng, lat, 1000, label='西塔', connect=True)
core.bubbles(az, sites, '用户数', hot=sites['site_id'].isin(top['site_id']), min_value=5, color='#C8CFD9',
             hot_color=MAIN, smax=1550, alpha=0.8, vmax=sites['用户数'].max(), ref=[500, 1000, 2000], title='气泡面积 = 连接用户数（人）')
viz.maps.circle(az, lng, lat, radius_m)
az.scatter([lng], [lat], marker='+', s=120, color=INK, lw=1.7, zorder=7)
ry = viz.maps.m2deg(radius_m, lat)[1]
az.text(lng, lat + ry * 1.08, f'西塔 {radius_m:,.0f} 米', ha='center', fontsize=11,
        color=MAIN, fontweight='bold', path_effects=viz.maps.HALO, zorder=7)
offsets = [(-35, 40), (40, 32), (-48, -32), (44, -35), (-65, 0)]
for (_, row), (dx, dy) in zip(top.iterrows(), offsets):
    if row['用户数'] < 5:
        continue
    az.annotate(f"{row['用户数']:,.0f} 人", (row['lng'], row['lat']), xytext=(dx, dy), textcoords='offset points', ha='right' if dx < 0 else 'left', va='center',
                fontsize=10, color=MAIN, fontweight='bold', path_effects=viz.maps.HALO, zorder=8, arrowprops=dict(arrowstyle='-', color=MAIN_MID, lw=0.8))
if poi is not None and len(poi):
    p = poi.copy()
    p['_distance'] = geo.haversine(p['lat'], p['lng'], lat, lng)
    nearest = p.loc[p['_distance'].idxmin()]
    az.annotate(f"{nearest['name']}\n距加权中心 {nearest['_distance']:.0f} 米", (lng, lat), xytext=(0.06, 0.12), textcoords='axes fraction',
                fontsize=9.5, color=INK, path_effects=viz.maps.HALO, zorder=9, arrowprops=dict(arrowstyle='-', color=INK, lw=0.8))
az.set_title('B  西塔周边 2×2 公里｜站点、中心、地标', loc='left', fontsize=11, pad=12)
top_note = '前五站点均在圈内' if top_in == 5 else f'前五站点有 {top_in} 个在圈内'
title = f'西塔 {radius_m:,.0f} 米圈触达 {reach_pct:.0%} 用户，{top_note}'
viz.finish(
    fig, title, f'左图为格内用户去重热力；右图为 2×2 公里放大，蓝色为前 5 站点，+ 为加权中心；圈内触达 {reach_n:,} 人\n气泡面积为站点连接人数，站点间有重复、不能相加；不足 5 人不画；触达按圈内任一站点去重；口径：dwd_signal × dim_cell', SOURCE, 'D4_02_西塔热点.png')
plt.show()


## 距离阈值变化一下，到过的人数会变成什么样子

每个人当天距离西塔最近的时候有多近？到过西塔500米以内的人占多少

In [ ]:
sig['到西塔(米)'] = geo.haversine(sig['lat'], sig['lng'],
                        hot_lat, hot_lng)
user_min = sig.groupby("imsi")['到西塔(米)'].min()
radius = pd.DataFrame({"半径(米)": [300, 500, 1000, 2000, 5000]})
radius['到过的用户占比'] = [(user_min < r_).mean() for r_ in radius['半径(米)']]
radius.to_sql("ads_hotspot_radius", con, if_exists="replace", index=False)

In [ ]:
# ── 出图 D4_03：ECDF ──
# 读者想问"多少人在 500 米以内"，这是累计问题：ECDF 直接读纵轴，直方图要靠眼睛把柱子加起来。
p500 = (user_min <= 500).mean()
fig, ax = plt.subplots(figsize=(10, 5.6))
ax.ecdf(user_min.clip(lower=10), color=MAIN, lw=2.2)
ax.set_xscale("log")                       # 距离跨三个数量级
ax.axvline(500, color=MUTED, lw=1, ls=(0, (4, 3)))
ax.annotate(f"500 米以内：{p500:.0%}", (500, p500), xytext=(10, -18), textcoords="offset points",
            fontsize=10.5, color=INK, fontweight="bold")
ax.scatter([500], [p500], color=MAIN, zorder=3)
ax.set_xticks([100, 300, 1000, 3000, 10_000, 30_000], ["100 米", "300 米", "1 公里", "3 公里", "10 公里", "30 公里"])
ax.yaxis.set_major_formatter("{x:.0%}")
ax.set_xlabel("当天离西塔最近的距离（对数刻度）", color=MUTED)
ax.grid(axis="x", visible=False)
viz.finish(fig, f"{p500:.0%} 的用户当天到过西塔 500 米以内",
           "曲线上一点 = 当天离西塔最近距离不超过横轴数值的用户占比；距离按所连站点的坐标计算；口径：dwd_signal × dim_cell",
           SOURCE, "D4_03_离西塔最近距离.png")
plt.show()

## 什么时候在的热点区域
每个小时有多少人在西塔500米以内？每个人第一次出现在西塔是在几点

In [ ]:
near = sig[sig['到西塔(米)'] <= 500]
near['ts'] =  pd.to_datetime(near['ts'],unit='ms')
hourly_near = near.groupby(near['ts'].dt.hour)['imsi'].nunique().reindex(
    range(24), fill_value=0)

first_n = near.groupby('imsi')['ts'].min().dt.hour.value_counts().reindex(
    range(24), fill_value=0)
xita_hourly = pd.DataFrame({'小时':range(24), "在场人数": hourly_near.values,
        "首次出现人数":first_n.values,
        "首次出现占比": (first_n / first_n.sum()).values}
        )
xita_hourly.to_sql("ads_xita_hourly", con, if_exists="replace", index=False)

In [ ]:
# x = pd.read_sql("SELECT * FROM ads_xita_hourly", con).set_index("小时")
new = x["首次出现人数"]
old = x["在场人数"] - new                      # 之前出现过、这一小时又在西塔的人
cum = new.cumsum()                             # 截至这一小时，累计到过西塔的人（去重）
V = int(cum.iloc[-1])                          # 到过西塔的总人数
n17_19 = int(new.loc[17:19].sum())
p17_19 = n17_19 / V
jump = new[17] / new[16]
pk = int(x["在场人数"].idxmax())
h = x.index.to_numpy()


# ── D4_04：完整在场时序 + 首次记录的时段占比 ──
x = xita_hourly.sort_values('小时').copy()
hours = x['小时'].to_numpy()
presence = x['在场人数'].to_numpy()
first = x['首次出现占比']
peak = x.loc[x['在场人数'].idxmax()]
p_window = first[x['小时'].between(17, 19)].sum()
part = np.array([first[x['小时'] <= 16].sum(), p_window, first[x['小时'] >= 20].sum()]) * 100
n16 = float(x.loc[x['小时'].eq(16), '在场人数'].iloc[0])
n17 = float(x.loc[x['小时'].eq(17), '在场人数'].iloc[0])
fig = plt.figure(figsize=(13.6, 6.6), layout='constrained')
gs = fig.add_gridspec(1, 2, width_ratios=[2.2, 1], wspace=0.13)
ax = fig.add_subplot(gs[0, 0])
detail = fig.add_subplot(gs[0, 1])
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(length=0, labelsize=10)
ax.grid(False)
ax.grid(axis='y', color='#E9EDF2', lw=0.7)
ax.set_axisbelow(True)
ax.axvspan(16.5, 19.5, color=MAIN_LIGHT, alpha=0.7, lw=0)
ax.fill_between(hours, presence, color=MAIN_LIGHT, alpha=0.38)
ax.plot(hours, presence, color=MAIN, lw=2.8)
ax.scatter(x.loc[x['小时'].isin([16, 17, int(peak['小时'])]), '小时'], x.loc[x['小时'].isin(
    [16, 17, int(peak['小时'])]), '在场人数'], s=45, color=MAIN, edgecolors='white', linewidths=1, zorder=5)
ax.annotate(f"{int(peak['小时'])} 点峰值\n{int(peak['在场人数']):,} 人", (peak['小时'], peak['在场人数']), xytext=(
    -6, 15), textcoords='offset points', ha='right', fontsize=11, color=MAIN_DARK, fontweight='bold')
ax.annotate(f'16 → 17 点\n{n16:,.0f} → {n17:,.0f} 人', (17, n17), xytext=(-70, 35), textcoords='offset points',
            ha='right', fontsize=10.5, color=INK, arrowprops=dict(arrowstyle='-', color=GRAY_DARK, lw=0.9))
ax.text(18, max(presence) * 0.1, '17–19 点', ha='center', fontsize=10, color=MAIN)
ax.set_ylim(0, presence.max() * 1.3)
ax.set_xlim(-0.5, 23.5)
ax.set_xticks([0, 4, 8, 12, 16, 20, 23], [f'{h} 点' for h in [0, 4, 8, 12, 16, 20, 23]])
ax.yaxis.set_major_formatter('{x:,.0f}')
ax.set_title('西塔 500 米内｜每小时记录到多少人', loc='left', fontsize=12, pad=17)
ax.set_ylabel('每小时去重人数（人）', fontsize=10, color=MUTED, labelpad=10)
for spine in detail.spines.values():
    spine.set_visible(False)
detail.tick_params(length=0, labelsize=10)
detail.grid(False)
detail.set_axisbelow(True)
detail.barh([0, 1, 2], part, height=0.48, color=[GRAY, MAIN, GRAY])
for yy, pp in enumerate(part):
    detail.text(pp + 1.5, yy, f'{pp:.0f}%', va='center', fontsize=14,
                fontweight='bold', color=MAIN if yy == 1 else INK)
detail.set_yticks([0, 1, 2], ['0–16 点', '17–19 点', '20–23 点'], fontsize=11)
detail.set_xlim(0, max(part) * 1.45)
detail.set_ylim(3.1, -0.85)
detail.set_xticks([])
detail.set_title('当天首次记录｜分别发生在哪一段', loc='left', fontsize=12, pad=17)
p0 = float(x.loc[x['小时'].eq(0), '首次出现占比'].iloc[0])
detail.text(0, 2.65, f'其中，0 点已经记录到 {p0:.0%}\n首次记录 ≠ 实际入场时刻',
            fontsize=10, color=MUTED, linespacing=1.7)
title = f"17–19 点贡献 {p_window:.0%} 的首次记录，{int(peak['小时'])} 点在场人数达 {int(peak['在场人数']):,}"
viz.finish(fig, title, '左：每小时在西塔 500 米内的去重人数；右：到过西塔的用户，按当天首次被记录的时段分组\n浅蓝底 = 17–19 点；时段长短不同，右图表示贡献占比；0 点的记录不证明前一晚未离开；口径：dwd_signal × dim_cell',
           SOURCE, 'D4_04_西塔在场与首次出现.png')
plt.show()

In [ ]:
# D1 的疑点：为什么 23 点信令最多？看各小时的信令有多少发生在西塔 1 公里内
sig['ts'] = pd.to_datetime(sig['ts'], unit='ms')
share_near = sig.assign(小时=sig["ts"].dt.hour, 近西塔=sig["到西塔(米)"]
                        <= 1000).groupby("小时")["近西塔"].mean()
print(f"10 点的信令 {share_near[10]:.0%} 发生在西塔 1 公里内；23 点是 {share_near[23]:.0%}")

## 样本口径：这些人是怎么被选出来的

- 97%的用户当天都到过和平区 （4.1）
- 86%的用户到过西塔500米以内，热点就在韩百商城，鑫韩城一带 （4.2， 4.3）
- 西塔的在场人数17点后猛增； 首次出现的时间在16:17点之间跳升过陡，不像自然的人流变化，更像从17点开始圈人（4.4）
-23点的信令大部分发生在西塔附近-D1的23点峰值这就不是全域作息，而是这批人晚上还在西塔

> 这份数据是2018年10月3日傍晚后到访沈阳西塔的用户，以及全天的信令，这是推断不是事实，交付的时候要写名，并想数据提供方确认画圈的条件

In [ ]:
# IMSI = 3位国家吗 + 2位运行商码 + 用户编号
mnc = sig.drop_duplicates('imsi')['imsi'].str[3:5].value_counts().rename_axis(
    "运行商代码").reset_index(name='用户数')
mnc['含义'] = mnc['运行商代码'].map({
    "00": '中国移动',
    "02": '中国移动',
    '07': '中国移动'
}).fillna('其他')
mnc

In [ ]:
quality.dq_record(con,
                  "DQ-15",
                  "代表性",
                  "样本圈选口径",
                  "dwd_signal",
                  f"97%的用户到过和平区，{p500:.0%} 到过西塔 500 米内",
                  "警告",
                  evidence="热点位于西塔；在场人数 17 点后猛增；首次出现时间 16→17 点跳升过陡",
                  decision="结论只针对“10 月 3 日傍晚后到访西塔的人群”；圈选条件需向数据提供方确认")
quality.dq_record(con,
                  "DQ-16",
                  "代表性",
                  "运营商覆盖",
                  "dwd_signal.imsi",
                  "全部为中国移动用户",
                  "警告",
                  evidence="IMSI 运营商代码只有 00、02、07",
                  decision="人数只代表移动用户；推算总人数需按移动市场份额扩样")

## 改写需求？这个能回答什么？不能回答什么？

具体来说是什么呢？2018年国庆第3天傍晚后，到到访西塔的人群，这里面都是中国移动的用户样本，他们几点来，待了多久，从哪来？还去了哪？那据此我们可以给西塔的夜间经济提出建议，那不能回答什么呢？

- 国庆夜间到访西塔的人几点来，待多久？几点走？ 不能回答什么？全市的夜间流量他不能回答，因为我们样本是按照西塔圈选的。

- 这些人从哪来？过夜地在哪个区？离西塔有多远？那不能做什么呢？工作日与节假日的对比，因为只有一天，而且还是假期，所以我们实际上分析很有限。

- 来西塔之前他们还去过哪些商圈，比如太原街、中街。那不能回答什么通勤职住分析，因为假期一般没有什么没有通勤。

- 西塔的客群能分成哪几类？ 但是不能做其他分析比如通勤方式。此外，联通、电信的用户情况，啊，总人数，因为我们只有移动用户，啊，需要按市场份额扩量，这个也做不到。

**这些能做什么，不能做什么，需要写入到需求改写文档中**我再重复一遍啊，企业里面这一步没做，后面的全是返工” 先把不能回答的都白纸黑字写下来，这比报告写的漂亮更重要

## 这群人白天在哪里？城市潮汐
样本是“傍晚到访西塔的人”。那他们白天在哪？如果白天也都在西塔附近，这份数据就只能讲西塔；如果白天散在全城，它还能讲“这群人的一天”。

做法：每人每小时取一个位置（这个小时所有信令位置的中位数：平均会落到几个站点中间的空地上）；先按区界切一刀：在不在和平区；再往下细分：和平区内分成西塔 500 米内和其他，和平区外分成其他 8 个城区和远郊县市（含市外），一共四类；最后每小时算各类的人数占比。

In [ ]:
# 每人每小时一个位置（该小时信令位置的中位数：平均会落到几个站点之间的空地上），先按区界分、再细分成四类
uh = sig.assign(
    hour=sig["ts"].dt.hour).groupby(
        ["imsi", "hour"],as_index=False)[["lng", "lat"]].median()
uh["district"] = pd.Series(
    geo.point_in_polygons(
        uh["lng"], uh["lat"],districts)).fillna("市外").to_numpy()
uh["dist_xita_m"] = geo.haversine(uh["lat"], uh["lng"], hot_lat, hot_lng)
CITY = ["沈河区", "皇姑区", "铁西区", "大东区", "浑南区", "于洪区", "沈北新区", "苏家屯区"]
order = ["西塔 500 米内", "和平区其他", "其他城区", "远郊县市及市外"]
in_hp = uh["district"] == "和平区"  # 第一刀：在不在和平区（零线就是它）
uh["区域"] = np.select(
    [in_hp & (uh["dist_xita_m"] <= 500), in_hp, uh["district"].isin(CITY)],
    order[:3],
    default=order[3])
cnt = uh.groupby(
    ["hour","区域"]).size().unstack(fill_value=0).reindex(
    index=range(24),columns=order,fill_value=0)
share = cnt.div(cnt.sum(axis=1), axis=0) * 100
share.round(1).loc[[4, 10, 15, 19, 23]]

In [ ]:
# 对账：人 × 小时的总数用范围判断（D03 重跑、清洗规则微调都会让它差几十条）；比例和峰值时刻才是要守住的结论
day_out = share.loc[10:16, ["其他城区", "远郊县市及市外"]].sum(axis=1).mean()
xs = share["西塔 500 米内"]
assert 86_000 < len(uh) < 87_000 and 70 < day_out < 73 and int(
    xs.idxmax()) == 19
print(
    f"✅ 对账通过：{len(uh):,} 个“人 × 小时”；白天 10–16 点 {day_out:.1f}% 在和平区以外；西塔占比 19 点最高（{xs.max():.1f}%）"
)

In [ ]:
# ── D4_05：沿用本稿四类区域，直接画 24 小时占比矩阵 ──
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle
groups = ['西塔 500 米内', '和平区其他', '其他城区', '远郊县市及市外']
counts = uh.groupby(['区域', 'hour']).size().unstack(
    fill_value=0).reindex(index=groups, columns=range(24), fill_value=0)
active = counts.sum(axis=0)
percent = counts.div(active.replace(0, np.nan), axis=1) * 100
peak_h = int(percent.loc[groups[0]].idxmax())
peak_pct = percent.loc[groups[0], peak_h]
key_hours = {4, 10, 15, 19, 23}
display = counts.copy()
display.index = ['西塔 500 米内', '和平区其他', '其他城区', '远郊县市 / 市外']
display.columns = [str(h) for h in range(24)]
fig = plt.figure(figsize=(14, 5.6), layout='constrained')
gs = fig.add_gridspec(1, 2, width_ratios=[1, 0.025], wspace=0.045)
ax, cax = (fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]))
vmax = max(0.1, np.ceil(percent.to_numpy().max() / 10) / 10)
matrix = viz.matrix(display, ax=ax, percent='col', totals=False, min_value=5, norm=Normalize(
    0, vmax), fontsize=10, rotate=0, row_title='', col_title='', diagonal='show')
for (row, col), text in matrix.texts.items():
    if int(col) not in key_hours and text.get_text() not in ('<5', '*'):
        text.set_text('')
for hh in key_hours:
    ax.add_patch(Rectangle((hh + 0.025, 0.025), 0.95, len(groups) - 0.05, fill=False,
                 ec=MAIN if hh == peak_h else '#A9B8C9', lw=1.7 if hh == peak_h else 0.8))
ax.set_xticks(np.arange(24) + 0.5, [f'{h}' for h in range(24)])
ax.tick_params(axis='x', labelsize=9, pad=9)
for hh, tick in enumerate(ax.get_xticklabels()):
    if hh in key_hours:
        tick.set(color=MAIN_DARK, fontweight='bold')
ax.set_title('每一列是一个小时，每一行是一类区域；颜色越深，该类区域占比越高', loc='left', fontsize=11, pad=19)
cb = fig.colorbar(matrix.mesh, cax=cax)
cb.outline.set_visible(False)
cb.ax.yaxis.set_major_formatter('{x:.0%}')
cb.ax.tick_params(length=0, labelsize=9)
cb.set_label('占该小时有记录用户的比例', fontsize=10, color=MUTED)
samples = ' · '.join((f'{h} 点 {active[h]:,} 人' for h in [4, 10, 15, 19, 23]))
ax.set_xlabel('小时（0–23 点）\n关键时刻分母：' + samples, fontsize=9.5,
              color=MUTED, labelpad=14, linespacing=1.8)
ax.xaxis.set_label_position('bottom')
rest10, restpk = (percent.loc[groups[2], 10], percent.loc[groups[2], peak_h])
title = f'{peak_h} 点西塔占比 {peak_pct:.0f}%；其他城区从 10 点 {rest10:.0f}% 降至 {restpk:.0f}%'
note = matrix.note
viz.finish(fig, title, '四类区域互斥、每列合计 100%；五个关键时刻标出具体百分数，全部小时共用同一色阶\n' + (note + '；' if note else '') +
           '每人每小时取信令位置中位数；分母是该小时有记录用户；口径：dwd_signal × dim_cell', SOURCE, 'D4_05_城市潮汐.png')
plt.show()

- 白天10-16点平均71.3%位于和平区外面，19点西塔占比最高为37.6%

In [ ]:
# ── D4_06：全市定位 + 五个时刻；pct、hp、SHOWN 沿用下一格对账 ──
from matplotlib.colors import BoundaryNorm
hs = dim_hotspot.loc[dim_hotspot["name"].eq("西塔")].iloc[0]
HOURS = [4, 10, 15, 19, 23]
SHOWN = ['和平区', '沈河区', '皇姑区', '大东区', '铁西区', '于洪区', '浑南区', '沈北新区', '苏家屯区']
act = uh.groupby('hour').size()
by_d = uh.groupby(['district', 'hour']).size().unstack(fill_value=0).reindex(
    index=[*districts, '市外'], columns=HOURS, fill_value=0)
pct = by_d.div(act.reindex(HOURS), axis=1) * 100
outside = 100 - pct.loc[SHOWN].sum()
bins = [0, 1, 2.5, 5, 10, 20, 40, 60]
cmap = viz.maps.heat_cmap().resampled(len(bins) - 1)
norm = BoundaryNorm(bins, cmap.N, clip=True)
at = {
    '和平区': (123.405, 41.756),
    '沈河区': (123.549, 41.814),
    '大东区': (123.518, 41.866),
    '皇姑区': (123.447, 41.877),
    '铁西区': (123.335, 41.8),
    '于洪区': (123.312, 41.884),
    '浑南区': (123.544, 41.729),
    '沈北新区': (123.455, 41.93),
    '苏家屯区': (123.31, 41.7),
}
alias = {
    '和平区': '和平',
    '沈河区': '沈河',
    '皇姑区': '皇姑',
    '大东区': '大东',
    '铁西区': '铁西',
    '于洪区': '于洪',
    '浑南区': '浑南',
    '沈北新区': '沈北',
    '苏家屯区': '苏家屯',
}
city_ = viz.maps.Regions(districts)
hidden = by_d.loc[SHOWN].lt(5)
fig = plt.figure(figsize=(14, 10.2), layout='constrained')
gs = fig.add_gridspec(3, 3, height_ratios=[1, 1, 0.075], hspace=0.06, wspace=0.04)
locator = fig.add_subplot(gs[0, 0])
axes = [fig.add_subplot(gs[i // 3, i % 3]) for i in range(1, 6)]
cax = fig.add_subplot(gs[2, :])
city_.base(locator, scale=dict(km=20, x=0.56, y=0.05), north=dict(x=0.92, y=0.97))
city_.fill(locator, colors={n: '#DCEAF5' if n in SHOWN else '#EEF0F3' for n in districts})
city_.edges(locator, ['和平区'], color=MAIN, lw=1.3)
city_.text(locator, dict.fromkeys(['新民市', '辽中区', '法库县', '康平县'],
           np.nan), fs=9, nameprops=dict(color=MUTED))
locator.set_title('全市定位｜方框为五张城区图的范围', loc='left', fontsize=11.5, pad=12)
for i, (ax, hh) in enumerate(zip(axes, HOURS)):
    city_.base(ax, scale=False, north=False, fill=False)
    values = pct[hh].drop('市外').copy()
    small_names = by_d.index[by_d[hh].lt(5)].difference(['市外'])
    values.loc[small_names] = np.nan
    pc = city_.fill(ax, values, norm=norm, cmap=cmap, edgecolor='white', lw=0.8)
    if len(small_names):
        small_regions = viz.maps.Regions({n: districts[n] for n in small_names})
        small_regions.fill(ax, colors='#FFFFFF', hatch='///', edgecolor=FAINT, lw=0.5)
    city_.edges(ax, ['和平区'], color=INK, lw=1.5)
    viz.maps.zoom(locator, ax, 123.425, 41.818, 14500, color=MAIN, connect=i == 0)
    if i == 0:
        viz.maps.scale_bar(ax, km=5, x=0.7, y=0.035)
    texts = {n: (alias[n], float(pct.loc[n, hh])) if not hidden.loc[n, hh]
             else (alias[n] + ' <5', np.nan) for n in SHOWN if n != '苏家屯区'}
    rt = city_.text(ax, texts, at=at, fs=9.5, fmt=lambda v: f'{v:.0f}%' if v >= 10 else f'{v:.1f}%')
    su = float(pct.loc['苏家屯区', hh])
    su_text = '<5' if hidden.loc['苏家屯区', hh] else f'{su:.1f}%'
    ax.annotate('苏家屯\n' + su_text, at['苏家屯区'], xytext=(0.09, 0.12), textcoords='axes fraction', ha='center', va='center',
                fontsize=9, color=INK, path_effects=viz.maps.HALO, zorder=9, arrowprops=dict(arrowstyle='-', color=GRAY_DARK, lw=0.7))
    if not hidden.loc['和平区', hh]:
        for text in (rt.texts['和平区'], rt.nums['和平区']):
            text.set(color='white', fontweight='bold', path_effects=[])
    ax.plot(float(hs['lng']), float(hs['lat']), marker='+', ms=8, mew=1.5, color='white', zorder=8)
    ax.set_title(f'{hh:02d}:00', loc='left', fontsize=14, color=INK, fontweight='bold', pad=12)
    ax.set_title(f'有记录 {act[hh]:,} 人\n外侧区县及市外 {outside[hh]:.1f}%',
                 loc='right', fontsize=9.3, color=MUTED, pad=12)
cb = fig.colorbar(pc, cax=cax, orientation='horizontal', ticks=bins, spacing='uniform')
cb.ax.set_xticklabels([f'{b:g}%' for b in bins], fontsize=9.5)
cb.outline.set_visible(False)
cb.ax.tick_params(length=0)
cb.set_label('区县人数 ÷ 该小时有记录的全部用户（分级色阶，五图共用）', fontsize=10, color=MUTED)
hp = pct.loc['和平区']
title = f'和平区占比从 10 点 {hp[10]:.0f}% 升至 23 点 {hp[23]:.0f}%，放大城区看变化'
viz.finish(fig, title, '每人每小时一个中位数位置；黑框 = 和平区，+ = 西塔；分母包含市外，颜色范围与分级在五图中完全一致\n外侧区县及市外 = 辽中、新民、法库、康平和市外，不等于几何方框外；不足 5 人的区以斜线和 <5 标识；口径：dwd_signal × dim_cell', SOURCE, 'D4_06_各时刻人在哪些区.png')
plt.show()